# Phi-2 QLoRA Fitness Caption Fine-Tuning

Attach a Kaggle Dataset containing `train.jsonl`, `val.jsonl`, and `finetune_phi2.py`, then run these cells from top to bottom.

## 1. Install dependencies

In [ ]:
!pip install -U transformers peft bitsandbytes accelerate datasets trl tqdm

## 2. Locate attached dataset files

If this cell cannot find your files automatically, set `DATASET_DIR` manually to the folder under `/kaggle/input/...` that contains the JSONL files.

In [ ]:
from pathlib import Path
import shutil

DATASET_DIR = None
for path in Path('/kaggle/input').glob('**/train.jsonl'):
    candidate = path.parent
    if (candidate / 'val.jsonl').exists() and (candidate / 'finetune_phi2.py').exists():
        DATASET_DIR = candidate
        break

if DATASET_DIR is None:
    raise FileNotFoundError('Could not find train.jsonl, val.jsonl, and finetune_phi2.py under /kaggle/input')

print(f'Using dataset directory: {DATASET_DIR}')
for name in ['train.jsonl', 'val.jsonl', 'finetune_phi2.py']:
    shutil.copy(DATASET_DIR / name, Path('/kaggle/working') / name)
    print(f'Copied {name}')

## 3. Train Phi-2 with QLoRA

In [ ]:
!python /kaggle/working/finetune_phi2.py \
  --train-file /kaggle/working/train.jsonl \
  --val-file /kaggle/working/val.jsonl \
  --output-dir /kaggle/working/phi2-caption-finetuned \
  --merged-output-dir /kaggle/working/phi2-caption-finetuned-merged

If the final merge runs out of memory, rerun training with `--skip-merge` or lower `--per-device-train-batch-size` to `2`.

## 4. Zip outputs for download

In [ ]:
!zip -r /kaggle/working/phi2-caption-finetuned.zip /kaggle/working/phi2-caption-finetuned
!if [ -d /kaggle/working/phi2-caption-finetuned-merged ]; then zip -r /kaggle/working/phi2-caption-finetuned-merged.zip /kaggle/working/phi2-caption-finetuned-merged; fi
!ls -lh /kaggle/working/*.zip